
# **PROJECT: County Health Rankings - West Virginia (2025)**

**Goal**
: Explore and prepare CHR&R data for rural/urban health disparity analysis (TruBridge project foundation).

## **Phase 1: Problem Definition and Data Requirements.**

### **Research Question:**

*Data Grounded*:
> 1. In rural West Virginia, how do hidden determinants such as social norms and health literacy mediate the relationship between structural social determinants of health and outcomes like chronic disease and behavioral health?

*TruBridge Smart Operations Focused*:
> 2 . How can understanding hidden determinants like social norms and health literacy help TruBridge anticipate care-seeking barriers and design smarter, more equitable healthcare operations in rural West Virginia?

### **Conceptual Framework:**

> **STRUCTURAL DETERMINANTS**
>
> *(poverty, education, rurality, broadband, access to care)*
>
>                              ** ↓ **
> -----
> -----
> **HIDDEN DETERMINANTS** (mediators)
>
> *(health literacy, social norms, preventive care behavior)*
>
>
> -----
> -----
>                              ** ↓ **
> **HEALTH OUTCOMES**
>
> *(chronic disease, unhealthy days, mortality, behavioral health)*
>
> -----
> -----

This project focuses on uncovering hidden determinants that influence rural health outcomes in West Virginia. By analyzing structural factors like poverty, education, and access to care alongside behavioral proxies for social norms and health literacy, the study aims to provide TruBridge with actionable insights for improving healthcare equity and operational efficiency through smart, data-driven interventions.

## **Phase 2: Data Collection and Access.**

###

**County Health Rankings & Roadmaps**

>     https://www.countyhealthrankings.org/

**County Health Rankings & Roadmaps - West Virigina Data and Resources**

>     https://www.countyhealthrankings.org/health-data/west-virginia/data-and-resources

### **Brief Summary**

> This project focuses on uncovering **hidden determinants** that influence rural health outcomes in West Virginia — particularly how **social norms and health literacy** interact with structural factors like poverty, education, and access to care.
>
>Using the **2025 County Health Rankings dataset**, I’m analyzing these relationships at the county level to identify patterns that help explain why some communities experience higher rates of chronic disease or behavioral health challenges despite similar structural conditions.
>
>The goal is not only to describe these disparities but to transform them into **actionable insights** that support TruBridge’s **Smart Operations** initiatives — meaning the use of data and analytics to help rural hospitals move from reactive care to proactive care.
>
>For example, if the data reveal that counties with lower broadband access and health literacy have higher preventable hospital stays, TruBridge could leverage those insights to **optimize telehealth deployment**, **target community education programs**, or **design predictive dashboards** that flag high-risk regions early.
>
>In short, the analytical work provides the “why” behind rural health disparities, and Smart Operations provides the “how” — how TruBridge can turn those insights into data-driven solutions that make healthcare more **equitable**, **efficient**, and **predictive**.

In [ ]:
# --- import libraries---
import io
import json, requests, plotly.express as px
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

from scipy.stats import pearsonr
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

from google.colab import drive
from google.colab import files

In [ ]:
# define base URL and file IDs

base = 'https://drive.google.com/uc?id='
main_id = '18OpPzeytrny8IkTpPZx2nFhiBq-aasIE'
add_id  = '1bOq-fkmoTT-W2kXw5rLLcPDCvN4ZUkjh'
sel_id  = '1vBA_LPnG2yvBFLK25W_iwog82NzTqBGe'

# Load datasets
df_main       = pd.read_excel(base + main_id)
df_additional = pd.read_csv(base + add_id, keep_default_na=False) # prevent text columns from turning in NaN incorrectly
df_select     = pd.read_csv(base + sel_id, keep_default_na=False)  # prevent text columns from turning in NaN incorrectly

# verify load success
print(" Main dataset shape:", df_main.shape)
print("Additional dataset shape:", df_additional.shape)
print("Select dataset shape:", df_select.shape)

In [ ]:
# peek at first few rows
df_main.head()

In [ ]:
# peek at first few rows
df_additional.head()

In [ ]:
# peek at first few rows
df_select.head()

In [ ]:
# clean up the Excel file which includes the intro text section

# peek at first 20 rows
pd.set_option('display.max_rows', 20)
df_main.head(20)

In [ ]:
# clean load for the main dataset

df_main = pd.read_excel(base + main_id, skiprows=1)

# verify (sanity check)
print(" Main dataset shape:", df_main.shape)
df_main.head()

In [ ]:
# Google Sheet ID (the long string in the link)
sheet_id = "18OpPzeytrny8IkTpPZx2nFhiBq-aasIE"

# Convert to XLSX download link
xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"

xls = pd.ExcelFile(xlsx_url)
print("Available sheet names:")
print(xls.sheet_names)

In [ ]:

# --- Load Tab 2: "Select Measure Data" ---
df_select = pd.read_excel(xlsx_url, sheet_name="Select Measure Data", skiprows=1)
print("Select Measure Data shape:", df_select.shape)
display(df_select.head())

In [ ]:
# --- Load Tab 4: "Additional Measure Data" ---
df_additional = pd.read_excel(xlsx_url, sheet_name="Additional Measure Data", skiprows=1)
print("Additional Measure Data shape:", df_additional.shape)
display(df_additional.head())

In [ ]:
# clean header text (hidden newlines/spaces) - polishes the labels on the dataset (column names)
def clean_columns(df):
  df = df.copy()
  df.columns = [str(c).strip().replace("\n", " ") for c in df.columns]
  return df

df_main = clean_columns(df_main)
df_select = clean_columns(df_select)
df_additional = clean_columns(df_additional)

In [ ]:
# basic cleanup (rows, columns, FIPS) - clean the data structure (rows, FIPS normalization, missing cleanup)

def basic_clean(df):
  # drop fully exmpy row/cols
  df = df.dropna(how="all").dropna(axis=1, how="all")
  # normalize FIPS as 5-char string (prevents losing zeros)
  if "FIPS" in df.columns:
    df["FIPS"] = (
            df["FIPS"].astype(str)
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"\D", "", regex=True)
            .str.zfill(5)
        )
    return df

df_select_clean = basic_clean(df_select.copy())
df_additional_clean = basic_clean(df_additional.copy())


# remove state totals & dedupe
df_select_clean     = df_select_clean.dropna(subset=["County"]).drop_duplicates(subset=["FIPS","County"])
df_additional_clean = df_additional_clean.dropna(subset=["County"]).drop_duplicates(subset=["FIPS","County"])

print("Select clean:", df_select_clean.shape)
print("Additional clean:", df_additional_clean.shape)

In [ ]:
df_select_clean

In [ ]:
# ensures no duplicate rows (counties or FIPs codes) in either dataset before merging
df_select_clean = df_select_clean.drop_duplicates(subset=["FIPS", "County"])
df_additional_clean = df_additional_clean.drop_duplicates(subset=["FIPS", "County"])

In [ ]:
# sanity check
print("Select clean:", df_select_clean.shape)
print("Additional clean:", df_additional_clean.shape)

In [ ]:
# save cleaned tabs (handy for re-use/colab)
df_select_clean.to_csv("select_measure_data_cleaned.csv", index=False)
df_additional_clean.to_csv("additional_measure_data_cleaned.csv", index=False)
print("Saved: select_measure_data_cleaned.csv, additional_measure_data_cleaned.csv")

In [ ]:
# Merge on shared keys
candidate_keys = ["FIPS", "State", "County"]
merge_keys = [k for k in candidate_keys if k in df_select_clean.columns and k in df_additional_clean.columns]
print("Using merge keys:",  merge_keys)

In [ ]:
# merge (outer = keep everything) + valiate - try strict one-to-one; if not, fall back (since de-duplicated, this should pass)
try:
  df_merged = pd.merge(
      df_select_clean, df_additional_clean,
      on=merge_keys, how="outer", validate="one_to_one"
  )

except Exception as e:
  print("Validatoin warning:", e)
  df_merged = pd.merge(df_select_clean, df_additional_clean, on=merge_keys, how='outer')

  # clean up merged
  df_merged = df_merged.dropna(subset=['County']).drop_duplicates(subset=merge_keys)

  print("Merged:", df_merged.shape)


In [ ]:
df_merged.head(3)

In [ ]:
# Sanity Check - Final Verification
terms = [
    "broadband",
    "unemploy",
    "unemployed",
    "life expectancy",
    "preventable",
    "insurance",
    "uninsured",
    "support",            # new: social/emotional support
    "english",            # new: language proficiency
    "physician",
    "poverty",
    "rural",
    "education"
]

for t in terms:
    hits = [c for c in df_merged.columns if t.lower() in c.lower()]
    print(f"\n🔍 Searching for '{t}' → {len(hits)} match(es)")
    if hits:
        print(" • " + "\n • ".join(hits[:12]))



In [ ]:
# 🧹 Standardized column names for analysis
df_merged = df_merged.rename(columns={
    # Employment
    "# Unemployed": "Num_Unemployed",
    "% Unemployed": "Percent_Unemployed",

    # Insurance coverage
    "# Uninsured Adults": "Num_Uninsured_Adults",
    "% Uninsured Adults": "Percent_Uninsured_Adults",
    "# Uninsured": "Num_Uninsured",
    "% Uninsured": "Percent_Uninsured",
    "# Uninsured Children": "Num_Uninsured_Children",
    "% Uninsured Children": "Percent_Uninsured_Children",

    # Broadband
    "# Households with Broadband Access": "Num_Broadband_Access",
    "% Households with Broadband Access": "Percent_Broadband_Access",

    # Language proficiency
    "# Not Proficient in English": "Num_Not_Proficient_English",

    # Social and emotional support
    "% Lacking Support": "Percent_Lacking_Support",

    # Access to care
    "Primary Care Physicians Ratio": "Primary_Care_Physicians_Ratio",
    "Preventable Hosp. Rate": "Preventable_Hospitalization_Rate",

    # Demographics
    "% Rural": "Percent_Rural",
    "% Adults with Some College": "Percent_Some_College",
    "% Poverty": "Poverty_Rate"
})



In [ ]:
print("Remaining columns with '%' or '#':")
[c for c in df_merged.columns if ('#' in c or '%' in c)]


In [ ]:
df_merged[['Population_x', 'Population_y']].describe()
(df_merged['Population_x'] - df_merged['Population_y']).abs().mean()


In [ ]:
df_merged = df_merged.rename(columns={
    'Population_x': 'Population_Total',
    'Population_y': 'Population_Total'
})


In [ ]:
df_merged['Population_Total'].isna().sum()


In [ ]:
# Save merged file
df_merged.to_csv("county_health_rankings_wv_merged.csv", index=False)
print("Saved: county_health_rankings_wv_merged.csv")

In [ ]:
print("Columns:", len(df_merged.columns))
print("Rows:", len(df_merged))
print(" \n Missing by column (top 20):")
print(df_merged.isna().sum().sort_values(ascending=False).head(20))


## **What We See So Far**

| Output Element             | What It Means                                                                  | Why It Matters                                                                                                                            |
| -------------------------- | ------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------- |
| **Columns: 415**           |  **415 measurable indicators** across West Virginia’s 55 counties. | This represents the full universe of potential *social determinants of health (SDOH)* — income, education, housing, health outcomes, etc. |
| **Rows: 55**               | Each row is a **county in West Virginia**.                                     | Perfect — this confirms you’re working at the right **geographic granularity** for TruBridge’s “rural health outcomes” objective.         |
| **Top 10 Missing Columns** | These are variables that have missing values in most counties.                 | It tells you where data suppression or limited measurement occurs (e.g., small population subgroups like “Hispanic injury death rate”).   |
| **dtype: int64**           | Just the data type of that summary list.                                       | Irrelevant for interpretation — it simply means those counts are integers.                                                                |



### **Interpretation in the TruBridge context**

| Insight                                                               | Relevance to Project                                                                                                                                                                                                                                                             |
| --------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **55 rows confirm full county coverage**                              | Every West Virginia county is represented — meaning the dataset can support statewide comparisons or clustering (important for rural vs. rural-adjacent analysis).                                                                                                              |
| **415 columns = very rich SDOH feature space**                        | This is the  “feature universe.” Later narrow it down to operationally relevant features — like education, access to care, broadband, health literacy proxies.                                                                                                            |
| **High missingness in race-specific or subgroup metrics**             | Important clue: Certain indicators (like “Hispanic Injury Death Rate”) are suppressed in small populations. This is typical in rural health data — it reflects **data sparsity**, not data error. TruBridge needs to know where data gaps exist when designing predictive tools. |
| **Patterns of missingness reveal where interventions may be limited** | Example: If smaller counties lack robust metrics for “preventable hospitalizations,” it signals poor data infrastructure — an insight for TruBridge’s Smart Operations dashboards.                                                                                               |



**Note**: **Focus analysis** on the strongest, complete indicators — things like:
* `Unemployment Rate`
* `Median Household Income`
* `% Adults with Some College`
* `Primary Care Physicians per 100k`
* `Preventable Hospitalization Rate`

These are the types of features to be used to train or visualize predictors of **care-seeking behavior** and **hospital performance**.

In [ ]:
# 1) Find any physician ratio columns
ratio_cols = [c for c in df_merged.columns
              if ("physician" in c.lower()) and ("ratio" in c.lower())]

# 2) Clean and coerce each to numeric
for col in ratio_cols:
    s = df_merged[col].astype(str).str.strip()
    s = s.str.replace(',', '', regex=False)      # remove thousands commas
    s = s.str.replace(':1', '', regex=False)     # remove the ":1" suffix
    s = s.replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    df_merged[col] = pd.to_numeric(s, errors='coerce')

# 3) Impute any remaining NaNs with the column median (safe for a few values)
for col in ratio_cols:
    if df_merged[col].isna().any():
        df_merged[col] = df_merged[col].fillna(df_merged[col].median())

# 4) Positivity check (now safely numeric)
for col in ratio_cols:
    nonpos = (df_merged[col] <= 0) | (~np.isfinite(df_merged[col]))
    if nonpos.any():
        print(f"⚠️  Non-positive/invalid values in '{col}':", int(nonpos.sum()))
    else:
        print(f"✅ '{col}' is numeric and strictly positive.")

# Optional: quick dtype confirmation
print("\nDtypes of ratio columns:")
print(df_merged[ratio_cols].dtypes)


In [ ]:
for c in [col for col in df_merged.columns if "physician" in col.lower() and "ratio" in col.lower()]:
    col_num = pd.to_numeric(df_merged[c], errors='coerce')
    if (col_num <= 0).any():
        print(f"⚠️  Non-positive values found in '{c}'.")
    else:
        print(f"✅ '{c}' has only positive values.")


In [ ]:
# ==============================================
# 🧩 FINAL SANITY CHECK — df_merged INTEGRITY (rev.)
# ==============================================
from IPython.display import display

print("🔍 Basic Structure")
print("-" * 40)
print(f"Rows: {len(df_merged)}")
print(f"Columns: {len(df_merged.columns)}")
print(f"Missing values (total): {int(df_merged.isna().sum().sum()):,}\n")

# --- 1️⃣ Key Columns / Themes Present? ---
key_terms = [
    "broadband",
    "unemploy",          # catches 'unemployment' & 'unemployed'
    "uninsured",
    "life expectancy",
    "preventable",       # preventable hospitalization
    "physician",         # primary care physicians ratio
    "poverty",
    "rural",
    "support",           # % lacking support
    "english",           # not proficient in english
    "population"         # population_total
]

print("🔎 Checking key feature presence")
print("-" * 40)
for term in key_terms:
    hits = [c for c in df_merged.columns if term.lower() in c.lower()]
    print(f"{term:<20}: {len(hits)} match(es)")
    if hits:
        print(" • " + "\n • ".join(hits[:12]))
    print()

# --- 2️⃣ Dataset Summary ---
print("📊 Data Types Summary")
print("-" * 40)
print(df_merged.dtypes.value_counts(), "\n")

print("📏 Numeric vs Non-numeric Columns")
print("-" * 40)
num_cols = df_merged.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns:     {len(num_cols)}")
print(f"Non-numeric columns: {df_merged.shape[1] - len(num_cols)}\n")

# --- 3️⃣ Missing Data Overview ---
print("🚨 Missing Values (Top 10 Columns)")
print("-" * 40)
missing_summary = df_merged.isna().sum().sort_values(ascending=False)
print(missing_summary.head(10), "\n")

# --- 4️⃣ Key Identifiers & Duplicates ---
print("🧾 Key Identifiers Check")
print("-" * 40)
if "FIPS" in df_merged.columns:
    print("Unique FIPS:", df_merged["FIPS"].nunique())
    print("Duplicate FIPS rows:", df_merged.duplicated(subset=["FIPS"]).sum())
if {"County","FIPS"}.issubset(df_merged.columns):
    print("Duplicate County+FIPS pairs:", df_merged.duplicated(subset=["County","FIPS"]).sum())
print()

# --- 5️⃣ Quick Range & Logic Checks ---
print("🧪 Range / Logic Checks")
print("-" * 40)
# Percent columns should be in [0,100]
percent_like = [c for c in df_merged.columns if c.lower().startswith("percent_") or c.strip().startswith("%")]
bad_pct = {}
for c in percent_like:
    s = df_merged[c]
    if s.dtype.kind in "if":
        mask = (s < 0) | (s > 100)
        if mask.any():
            bad_pct[c] = int(mask.sum())
if bad_pct:
    print("⚠️ Percent columns with values outside [0,100]:", bad_pct)
else:
    print("✅ All percent-like columns within [0,100].")

# Physicians ratio should be positive
for c in [col for col in df_merged.columns if "physician" in col.lower() and "ratio" in col.lower()]:
    if (df_merged[c] <= 0).any():
        print(f"⚠️ Non-positive values found in '{c}'.")
    else:
        print(f"✅ '{c}' has only positive values.")
print()

# --- 6️⃣ Must-have fields exist? (post-renaming names)
must_have = [
    "Population_Total",
    "Percent_Rural",
    "Percent_Broadband_Access",
    "Poverty_Rate",
    "Percent_Unemployed",
    "Primary_Care_Physicians_Ratio",
    "Preventable_Hospitalization_Rate",
    "Percent_Lacking_Support",
    "Num_Not_Proficient_English"  # will create rate below if present
]
missing_must = [m for m in must_have if m not in df_merged.columns]
print("📌 Must-have fields missing:", missing_must if missing_must else "None", "\n")

# --- 7️⃣ Create rate for English proficiency if needed ---
if ("Num_Not_Proficient_English" in df_merged.columns) and ("Population_Total" in df_merged.columns):
    # Drop the duplicate 'Population_Total' column before calculating the rate
    df_merged = df_merged.loc[:,~df_merged.columns.duplicated()].copy()
    rate_col = "Percent_Not_Proficient_English"
    if rate_col not in df_merged.columns:
        df_merged[rate_col] = (df_merged["Num_Not_Proficient_English"] / df_merged["Population_Total"]) * 100
        print(f"✨ Created {rate_col} from counts / population.\n")

# --- 8️⃣ Quick Preview ---
print("👀 Data Preview")
print("-" * 40)
display(df_merged.head(3))

In [ ]:
# Sanity Check
dupes = df_merged.columns[df_merged.columns.duplicated()].tolist()
print("Duplicate column names:", dupes if dupes else "None — all clear ✅")


## **Phase 3 - Data Cleaning and Preparation**

### **Identify Missing Values**

In [ ]:
# ===============================================
# 🔍 Phase 3: Data Cleaning — Identify Missing Values
# ===============================================

# 1️⃣ Total count of missing values
total_missing = df_merged.isna().sum().sum()
print(f"🧮 Total Missing Values: {total_missing:,}\n")

# 2️⃣ Missing values per column (sorted)
missing_summary = (
    df_merged.isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'index': 'Column', 0: 'Missing_Count'})
)

# 3️⃣ Filter columns with at least 1 missing value
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0]

# 4️⃣ Display top 15 columns with most missing data
print("📊 Top Columns with Missing Data:")
display(missing_summary.head(15))

# 5️⃣ Quick percentage context (optional but helpful)
missing_summary['Percent_Missing'] = (
    missing_summary['Missing_Count'] / len(df_merged) * 100
)
print("\n📈 Overall Missing Value Summary:")
display(missing_summary.head(15))


In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df_merged.isnull(), cbar=False)
plt.title("Missing Value Map - df_merged")
plt.show()

***Analysis***:

Represents the columns with the most missing data `Injury Death Rate (Hispanic)`, `YPLL Rate (Hispanic)`, `Preventable Hosp. Rate (Asian)` Each number (mostly 53 or 54) means that out of 55 West Virginia counties, data is missing for 53-54 of of them which is approximately (≈) 95-98 % missingness.

***Relevance***:

These are *not random gaps* - they show **data sparsity** for minority for low-population groups. *Structural Truth*: Some SDOH indicators simply aren't measured everywhere. This helps TruBridge interpret not only *health outcomes* but also *data coverage inequity*.

In [ ]:
# 🔎 MISSING VALUES DIAGNOSTIC REPORT
# ==============================================

# 1️⃣ Count total missing values
total_missing = df_merged.isna().sum().sum()
percent_missing = round((total_missing / (df_merged.shape[0] * df_merged.shape[1])) * 100, 2)

print(f"📊 Total Missing Values: {total_missing:,}")
print(f"📈 Percent Missing (Overall): {percent_missing}%\n")

# 2️⃣ Identify which columns have missing data
missing_summary = (
    df_merged.isna()
        .sum()
        .to_frame("Missing_Count")
        .assign(Missing_Percent=lambda x: round((x["Missing_Count"] / len(df_merged)) * 100, 2))
        .sort_values(by="Missing_Count", ascending=False)
)

# Filter to only those with missing data
missing_summary = missing_summary[missing_summary["Missing_Count"] > 0]

print("Top 15 Columns with Missing Data:\n")
display(missing_summary.head(15))

# 3️⃣ Quick group summary by level of missingness
bins = [0, 10, 25, 50, 75, 100]
labels = ["<10%", "10–25%", "25–50%", "50–75%", ">75%"]
group_summary = (
    pd.cut(missing_summary["Missing_Percent"], bins=bins, labels=labels, include_lowest=True)
        .value_counts()
        .sort_index()
)

print("\nColumns by Missingness Range:\n", group_summary)


In [ ]:
# ===============================================
#  Next Step: Visual + Strategic Overview
# ===============================================

# 1️⃣ Visualize top missing columns
plt.figure(figsize=(10,6))
sns.barplot(
    data=missing_summary.head(15),
    x="Missing_Percent",
    y=missing_summary.head(15).index,
    palette="coolwarm"
)
plt.title("Top 15 Columns with Missing Data (%)")
plt.xlabel("Percent Missing")
plt.ylabel("Feature")
plt.show()

# 2️⃣ Quick Recommendation Summary
print("🧭 Recommended Actions by Missingness Range:")
print("- Features < 10% missing → ✅ Impute (Median or Mean)")
print("- Features 10–25% missing → ⚖️ Impute with caution (Median preferred)")
print("- Features 25–50% missing → ⚠️ Consider domain review or partial drop")
print("- Features > 50% missing → 🚫 Drop or flag for further investigation")



#### **Impute Missing Values Using Median**

In [ ]:
# ===============================================
# PHASE 3.4 — Selective Median Imputation (Robust)
# ===============================================

# 0) Work on a copy
df_imputed = df_merged.copy()

# 1) Never impute these (identifiers / text)
exclude_cols = {'FIPS', 'County', 'State', 'County_Name'}
exclude_cols = {c for c in exclude_cols if c in df_imputed.columns}

# 2) Coerce any numeric-looking columns to numeric (safe)
#    (This catches things like "3,094" or stray strings.)
for col in df_imputed.columns:
    if col in exclude_cols:
        continue
    # try a cheap numeric probe; if it works, keep numeric dtype
    coerced = pd.to_numeric(df_imputed[col], errors='coerce')
    # Only replace if it meaningfully increased numeric coverage
    if coerced.notna().sum() >= df_imputed[col].notna().sum() * 0.8:
        df_imputed[col] = coerced

# 3) Choose only numeric columns with < 50% missing for median impute
numeric_cols = df_imputed.select_dtypes(include=[np.number]).columns.difference(exclude_cols)
missing_rate = df_imputed[numeric_cols].isna().mean()
valid_impute_cols = missing_rate[missing_rate < 0.50].index.tolist()

# 4) Median impute (column-wise)
df_imputed[valid_impute_cols] = df_imputed[valid_impute_cols].fillna(
    df_imputed[valid_impute_cols].median()
)

# 5) OPTIONAL: drop very sparse numeric columns (>= 50% missing)
drop_sparse = missing_rate[missing_rate >= 0.50].index.tolist()
# If you want to keep them for now, comment the next line:
# df_imputed = df_imputed.drop(columns=drop_sparse)

# 6) Categorical columns: fill NaNs with mode (if any exist)
cat_cols = df_imputed.select_dtypes(exclude=[np.number]).columns.difference(exclude_cols)
for c in cat_cols:
    if df_imputed[c].isna().any():
        try:
            df_imputed[c] = df_imputed[c].fillna(df_imputed[c].mode(dropna=True).iloc[0])
        except Exception:
            # If no mode (all NaN), drop or leave as-is
            pass

# 7) Diagnostics
remaining_total = int(df_imputed.isna().sum().sum())
leftover_cols = df_imputed.columns[df_imputed.isna().any()].tolist()

print("✅ Selective median imputation complete.")
print(f"   Imputed numeric columns (<50% missing): {len(valid_impute_cols)}")
print(f"   Sparse numeric columns (>=50% missing): {len(drop_sparse)}  -> {drop_sparse[:8]}{' ...' if len(drop_sparse)>8 else ''}")
print(f"   Remaining missing values (all cols): {remaining_total}")
if leftover_cols:
    print(f"   Columns still containing NaNs: {leftover_cols[:12]}{' ...' if len(leftover_cols)>12 else ''}")

# 8) Save snapshot
out_path = '/content/df_imputed_phase3.csv'
df_imputed.to_csv(out_path, index=False)
print(f"💾 Saved: {out_path}")


In [ ]:
# ===============================================
# PHASE 3.5 — Final Cleanup Before EDA
# ===============================================

# Reload if needed
df_imputed = pd.read_csv("/content/df_imputed_phase3.csv")

# Drop columns with >50% missing values
df_final = df_imputed.dropna(axis=1, thresh=len(df_imputed)*0.5)

# Verify
print(f"✅ Final dataset shape: {df_final.shape}")
print(f"Remaining missing values: {int(df_final.isna().sum().sum())}")

# Save cleaned version
df_final.to_csv("/content/df_final_cleaned.csv", index=False)
print("💾 Saved: /content/df_final_cleaned.csv")


#### **Additional Feature - Sparsity as a Signal**

In [ ]:
# ===============================================
# PHASE 3.6 — Post-Imputation Diagnostics & New Feature
# ===============================================

# 1️⃣ Confirm no missing values remain
plt.figure(figsize=(14,6))
sns.heatmap(df_final.isnull(), cbar=False, cmap=sns.color_palette(["#27AE60", "#E74C3C"]))
plt.title("Post-Imputation Missing Data Heatmap", fontsize=12, fontweight="bold")
plt.show()

# 2️⃣ Add 'Data Coverage' feature (sparsity as a signal)
df_final["Data_Coverage_Score"] = df_final.notna().sum(axis=1)
df_final["Data_Coverage_Percent"] = (
    df_final["Data_Coverage_Score"] / df_final.shape[1] * 100
).round(2)

# 3️⃣ Quick preview
display(df_final[["County", "Data_Coverage_Score", "Data_Coverage_Percent"]].head())

# 4️⃣ Optional cleanup
df_final = df_final.copy()
print("✅ DataFrame defragmented successfully.")


df_final.loc[:, "Data_Coverage_Score"] = df_final.notna().sum(axis=1)
df_final.loc[:, "Data_Coverage_Percent"] = (
    df_final["Data_Coverage_Score"] / df_final.shape[1] * 100
).round(2)


In [ ]:
# ---------- County-level data coverage plot (robust) ----------

df_plot = df_final.sort_values("Data_Coverage_Percent", ascending=False).copy()

# If everything is 100%, a plot may be redundant—handle gracefully
if np.isclose(df_plot["Data_Coverage_Percent"], 100).all():
    print("✅ All counties have 100% Data Coverage after cleaning—plot is optional.")
else:
    # dynamic height: ~0.25 inch per county (min 6 inches)
    height = max(6, 0.25 * len(df_plot))
    cmap = sns.color_palette("crest", as_cmap=True)

    plt.figure(figsize=(12, height))
    ax = sns.barplot(
        data=df_plot,
        y="County",
        x="Data_Coverage_Percent",
        hue="Data_Coverage_Percent",   # avoids FutureWarning
        palette=cmap,
        dodge=False,
        legend=False
    )

    ax.set_title("County-Level Data Coverage (%) — CHR&R 2025 (West Virginia)",
                 fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Percent of Variables Reported", fontsize=12)
    ax.set_ylabel("County", fontsize=12)
    ax.set_xlim(0, 100)
    ax.grid(axis="x", linestyle="--", alpha=0.5)

    # annotate bars
    for p in ax.patches:
        ax.text(p.get_width() + 0.6,               # a little to the right of the bar
                p.get_y() + p.get_height() / 2,
                f"{p.get_width():.1f}%",
                va="center", fontsize=9)

    plt.tight_layout()
    plt.savefig("/content/data_coverage_by_county.png", dpi=180, bbox_inches="tight")
    plt.show()
    print("💾 Saved figure: /content/data_coverage_by_county.png")



#### **Side-by-Side Pre vs. Post Imputation**

In [ ]:
# Optional: Validation step (Pre vs. Post Imputation Visualization)
# Keep this for transparency if preparing a final report or technical appendix.
# Safe to skip during regular workflow.

In [ ]:
# copy pre-imputation dataset
df_reduced = df_merged.copy()

In [ ]:
# Columns to visualize
compare_columns = ['Median Household Income', 'Life Expectancy']

In [ ]:
# loop through Side-by-side distribution plots
for col in compare_columns:
    if col in df_reduced.columns and col in df_imputed.columns:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

        # Before imputation
        axes[0].hist(df_reduced[col].dropna(), bins=15, edgecolor='black', color='#E57373', alpha=0.8)
        axes[0].set_title(f"{col} — Before Imputation", fontsize=11, fontweight="bold")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Number of Counties")

        # After imputation
        axes[1].hist(df_imputed[col], bins=15, edgecolor='black', color='#64B5F6', alpha=0.8)
        axes[1].set_title(f"{col} — After Median Imputation", fontsize=11, fontweight="bold")
        axes[1].set_xlabel(col)

        fig.suptitle(f"{col}: Distribution Before vs. After Imputation", fontsize=13, fontweight="bold", y=1.05)
        plt.tight_layout()
        plt.show()
    else:
        print(f" Column '{col}' not found in dataset.")

#### **Check and Handle Duplicates**

In [ ]:
# Check for duplicate rows
duplicate_count = df_imputed.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")


In [ ]:
# sanity check - confirm one record per county (since WV has 55 counties)
print(f"Unique counties: {df_imputed['County'].nunique()}")
print(f"Total rows: {len(df_imputed)}")


In [ ]:
# example normalization on a single column

# income = df_imputed['Median Household Income']
# normalized_income = (income - np.min(income)) / (np.max(income) - np.min(income))

# print("Normalized income (0–1 range):")
# print(normalized_income.head())

In [ ]:
# export the cleaned data, as a backup external to google colab
df_imputed.to_csv("wv_trubridge_cleaned_phase3.csv", index=False)
print("Saved: county_health_rankings_wv_merged_clean.csv")

In [ ]:
# download to desktop
files.download("wv_trubridge_cleaned_phase3.csv")

In [ ]:
# dataset overview

print(df_final.info())
display(df_final.describe().T.head(10))
print(f"\nRows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")



In [ ]:
key_features = [
    # 🏗️ Structural SDOH
    "% Children in Poverty",                # proxy for Poverty Rate
    "Percent_Unemployed",                   # confirmed
    "Percent_Broadband_Access",             # confirmed
    "% Some College",                       # confirmed

    # 🧠 Behavioral / Hidden Determinants
    "Social Association Rate",              # confirmed
    # Health Literacy missing — may use proxy later (e.g., "% Adults with Some College" or Education Index)

    # ❤️ Health Outcomes
    "Preventable Hospitalization Rate",     # confirmed
    "Life Expectancy",                      # confirmed

    # 🧮 Data Integrity / Sparsity Signal
    "Data_Coverage_Percent"                 # confirmed
]



In [ ]:
existing = [col for col in key_features if col in df_final.columns]
missing = [col for col in key_features if col not in df_final.columns]

print("✅ Existing columns:", existing)
print("⚠️ Missing columns:", missing)


In [ ]:
keywords = ["poverty", "employ", "broadband", "college", "literacy"]
for word in keywords:
    matches = [col for col in df_final.columns if word in col.lower()]
    print(f"\n🔍 Matches for '{word}':")
    print(matches)


In [ ]:
# Sparsity as a Signal: Data Completeness vs Health Outcomes
plt.figure(figsize=(10,7))

sns.scatterplot(
    data=df_final,
    x="Data_Coverage_Percent",              # completeness signal
    y="Preventable Hospitalization Rate",   # outcome
    hue="% Children in Poverty",            # structural SDOH
    size="Percent_Broadband_Access",        # digital access proxy
    palette="coolwarm",
    alpha=0.8,
    sizes=(40, 300)
)

plt.title(
    "Sparsity as a Signal: Data Coverage vs Preventable Hospitalizations\nCHR&R 2025 – West Virginia",
    fontsize=13, fontweight="bold", pad=15
)
plt.xlabel("Data Coverage (%)", fontsize=11)
plt.ylabel("Preventable Hospitalization Rate", fontsize=11)

plt.legend(title="Legend", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='both', linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


#### **Plot 1: “Sparsity as a Signal"**
- What Plot 1 shows:

This chart compares:
- **X-axis**: `Data Coverage (%)` — how complete each county’s dataset is.
- **Y-axis**: `Preventable Hospitalization Rate` — health outcome (how often people are hospitalized for preventable issues).
- **Color**: % Children in Poverty — economic burden indicator.
- **Size**: Percent Broadband Access — digital access proxy.

Analysis: At this juncture the data is clean - every county is reprsented equally. Any variation in hospitalization now reflects real-world differences, not missing data. Some West Virginia counties share similar preventable hospitalization rates — and others are distinctly higher or lower. No one falls in between. It's a sign of patterned inequality, not missing information.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df_final["Preventable Hospitalization Rate"], bins=15, kde=True, color="#64B5F6")
plt.title("Distribution of Preventable Hospitalization Rates (West Virginia)")
plt.xlabel("Hospitalization Rate")
plt.ylabel("Number of Counties")
plt.show()


**Analysis**:
- Most counties (between ~2,000 and 4,500) have moderately low preventable hopsitalization rates. Extending out toward 10,000 — only a few counties with very high hospitalization rates.  

Most West Virginia counties report relatively low rates of preventable hospitalizations, but a handful of counties have extremely high rates — possibly due to poorer healthcare access, chronic illness prevalence, or socioeconomic stress.

Those high-rate counties are outliers, but they’re not random — they often align with rural, lower-income areas with fewer healthcare facilities.

The right tail represents your “signal.”
This is where your “Sparsity as a Signal” concept shines — these outlier counties are:
- Sparse in population or healthcare infrastructure,
- Overrepresented in hospital visits, and
- Critical targets for TruBridge’s proactive intervention models.


In [ ]:
# Plot 2: “Preventable Hospitalizations vs. Poverty and Education”
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=df_final,
    x="% Children in Poverty",
    y="Preventable Hospitalization Rate",
    hue="% Some College",
    size="Percent_Broadband_Access",
    palette="viridis",
    alpha=0.8
)

plt.title("Preventable Hospitalizations vs Poverty and Education", fontsize=14, fontweight="bold")
plt.xlabel("% Children in Poverty", fontsize=12)
plt.ylabel("Preventable Hospitalization Rate", fontsize=12)

# ✅ Move legend outside the plot
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    borderaxespad=0,
    title="Legend"
)

plt.tight_layout()
plt.show()


#### **Plot 2: “Preventable Hospitalizations vs Poverty and Education"**

**Analysis**
- X-axis: `% Children in Poverty` → a proxy for economic disadvantage.
- Y-axis: `Preventable Hospitalization Rate` → how often people go to the hospital for preventable conditions.
- `Color (hue)`: % Some College → a proxy for education / health literacy level.
- `Bubble size`: Percent Broadband Access → a proxy for information and resource access.

Counties with more children in poverty tend to have higher preventable hospitalization rates.

First empirical validation of a known SDOH link: When people live in poverty, they tend to get less preventive healthcare (like checkups or screenings). Because they do not get early care, they often end up needing more hospital visits later on.

- **General Trend**: Counties with more children in poverty tend to have higher preventable hospitalization rates.
- **Education**: Education (as a proxy for health literacy) acts as a protective factor — more education → fewer preventable hospitalizations.
- **Broadband access**: Broadband access may indirectly improve health outcomes by improving access to telehealth, information, and scheduling resources — even in rural regions.

Counties with higher poverty, lower education, and limited broadband access experience significantly higher rates of preventable hospitalizations — reinforcing the importance of addressing social and informational inequities in West Virginia’s rural health landscape.



The "Smart Operations" story, so far...

What this plot reveals is a **multivariate relationship** that can drive TruBridge’s data strategy:
- **High poverty** → higher hospitalization risk
- **Low education** → amplifies the risk
- **Limited broadband** → further reduces prevention and outreach potential

That means these counties are **prime candidates** for:
- Predictive flagging models
- Targeted literacy and telehealth interventions
- Resource allocation optimization

## **Phase 4 — Exploratory Data Analysis (EDA)**

***Purpose**: uncover "hidden determinants" that shape rural health in West Virginia.*

***Focus:*** Curate a focused subset of variables that actually speak to the purpose of uncovering hidden determinants that shape rural health in West Virginia

| Theme                                  | Example Columns                                                                                                                           | Why It Matters                                  |
| -------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------- |
| **Health Outcomes**                    | `Deaths`, `Years of Potential Life Lost Rate`, `Average Number of Physically Unhealthy Days`, `Average Number of Mentally Unhealthy Days` | Direct measure of population health status.     |
| **Economic Determinants**              | `Median Household Income`, `Unemployment Rate`, `% Children in Poverty`, `% Uninsured`                                                    | Financial access to care and quality of life.   |
| **Educational & Social Determinants**  | `% Adults with Some College`, `% High School Graduation`, `Social Association Rate`, `% Children in Single-Parent Households`             | Indicators of community resilience and support. |
| **Behavioral & Clinical Risk Factors** | `% Smokers`, `% Obese`, `% Physically Inactive`, `% Access to Exercise Opportunities`                                                     | Behavior-linked health outcomes.                |
| **Rurality & Infrastructure**          | `% Rural Residents`, `Population`, `Primary Care Physicians Ratio`, `Broadband Access`, `Food Environment Index`                          | Hidden access-related factors for rural health. |


In [ ]:
# sanity check for df_final dataframe
df_final.head()

In [ ]:
df_final.head()

In [ ]:
# structure and data types / general information
df_final.info()

In [ ]:
# Check total number of columns
print(f"Total columns: {len(df_final.columns)}\n")

# Display a preview of all column names
all_columns = df_final.columns.tolist()

# Let's search by key themes
themes = {
    "health": ["death", "life lost", "physically", "mentally", "unhealthy"],
    "economic": ["income", "poverty", "unemployment", "uninsured", "employment"],
    "education_social": ["education", "college", "high school", "single", "association", "social"],
    "behavioral": ["smok", "obese", "exercise", "inactive", "alcohol"],
    "rural_infra": ["rural", "broadband", "care", "physician", "access", "transport"]
}

# Print out matches for each theme
for theme, keywords in themes.items():
    print(f"\n {theme.upper()}-RELATED COLUMNS:")
    matches = [col for col in all_columns if any(k.lower() in col.lower() for k in keywords)]
    for m in matches:
        print("  -", m)


In [ ]:
# create an EDA subset
# Focused EDA subset for uncovering hidden determinants of rural health
focus_cols = [
    # --- Health Outcomes ---
    'Deaths',
    'Years of Potential Life Lost Rate',
    'Average Number of Physically Unhealthy Days',
    'Average Number of Mentally Unhealthy Days',

    # --- Economic Determinants ---
    'Median Household Income',
    '% Children in Poverty',
    'Income Ratio',
    'Percent_Uninsured',
    '80th Percentile Income',

    # --- Education & Social Determinants ---
    '% Some College',
    '% Completed High School',
    'Social Association Rate',
    '% Children in Single-Parent Households',

    # --- Behavioral Factors ---
    '% Adults Reporting Currently Smoking',
    '% Physically Inactive',
    '% With Access to Exercise Opportunities',

    # --- Rurality & Infrastructure ---
    'Percent_Broadband_Access',
    '# Primary Care Physicians',
    'Primary Care Physicians Rate',
    'Percent_Rural'
]

# Subset the dataset
df_focus = df_final[focus_cols]

# Verify
print(f" Focused subset shape: {df_focus.shape}")
df_focus.head()


##### **Phase 4.1 — Summary Statistics & Correlation Insight**

In [ ]:
# structural overview - general information
print("=== Basic Info ===")
df_focus.info()

In [ ]:
print("\n=== Missing Values ===")
missing = df_focus.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

In [ ]:
# descriptive statistics
print("\n=== Descriptive Statistics ===")
df_focus.describe().T

**Correlation Matrix (Numerical Relationships)**

In [ ]:
# compute correlations
corr = df_focus.corr(numeric_only=True)

# Plot
plt.figure(figsize=(12,10))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap — Rural Health Determinants", fontsize=14)
plt.show()

**Hidden Determinants Emerging Clearly**

These correlations validate the study’s core premise:

> *Structural factor* — poverty, education, healthcare access, and rural isolation — are intertwined and quietly drive disparities in health outcomes.

**Hidden determinants here are:**
- Educational attainment
- Broadband/infrastructure gaps
- Rural density
- Access to primary care

These factors don’t directly cause illness but mediate health-seeking behavior and care access — aligning perfectly with the research goal of helping TruBridge hospitals transition from reactive to proactive care.

##### **Phase 4.2 — Visual Storytelling EDA**

In [ ]:
# Distribution Analysis — Inequality & Skew

# Choose variables by theme
dist_cols = [
    'Years of Potential Life Lost Rate',
    '% Children in Poverty',
    'Median Household Income',
    '% Some College',
    '% Physically Inactive',
    'Percent_Broadband_Access',
    'Percent_Rural'
]

# Plot histograms
plt.figure(figsize=(14,10))
for i, col in enumerate(dist_cols, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df_focus[col], kde=True, color='steelblue')
    plt.title(col, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate Scatterplots — Determinants vs Outcomes

plt.figure(figsize=(16,9))

pairs = [
    ('% Children in Poverty', 'Years of Potential Life Lost Rate'),
    ('Percent_Broadband_Access', 'Average Number of Mentally Unhealthy Days'),
    ('% Some College', '% Adults Reporting Currently Smoking'),
]

for i, (x, y) in enumerate(pairs, 1):
    plt.subplot(1, 3, i)
    sns.regplot(data=df_focus, x=x, y=y, scatter_kws={'alpha':0.7})
    plt.title(f"{x} vs {y}", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Correlations Within Infrastructure & Behavior

infra_cols = [
    'Percent_Rural',
    'Percent_Broadband_Access',
    '# Primary Care Physicians',
    'Primary Care Physicians Rate',
    '% Physically Inactive',
    '% Adults Reporting Currently Smoking'
]

plt.figure(figsize=(8,6))
sns.heatmap(df_focus[infra_cols].corr(), annot=True, cmap='vlag', center=0)
plt.title("Infrastructure & Behavior Interplay", fontsize=12)
plt.show()


**Proven Analysis So Far**

The EDA visual confirm the structural scaffolding or framework of rural health disparities, meaning the EDA provide clear evidence of the underlying foundational factors, that create and maintain differences in health outcomes between rural and non-rural populations.

| Structural Determinants                 | Correlated Outcomes                              |
| --------------------------------------- | ------------------------------------------------ |
| Poverty, low income                     | ↑ Years of Potential Life Lost, ↑ Unhealthy Days |
| Low education, single-parent households | ↑ Smoking, ↑ Inactivity                          |
| Rurality, low broadband, fewer PCPs     | ↓ Preventive access, ↑ behavioral risks          |

So the foundation is clean: structural variables shape the conditions under which people life.


**Insight:**
> Structural disadvantages like poverty, education, and rural isolation do not act alone — they influence hidden social and behavioral mediators such as health literacy, social norms, and preventive care behaviors. These mediators, in turn, shape measurable health outcomes (mortality, unhealthy days).

**Why it matters for TruBridge:**
> Identifying and tracking these hidden mediators enables hospitals to target behavioral outreach and digital access programs — not just economic aid — to improve community health outcomes.


##### **Phase 4.3 - Exploratory Discovery EDA**


In [ ]:
# relationships among key social and economic variables
plt.figure(figsize=(12,6))
sns.pairplot(df_focus[
    ['Median Household Income', '% Children in Poverty', '% Some College', '% Completed High School', 'Percent_Broadband_Access']
], diag_kind='kde', corner=True)
plt.suptitle("Interplay of Economic and Social Determinants", y=1.02, fontsize=14)
plt.show()


In [ ]:
# outcome vs Behavioral Determinants (Dual Influence)
plt.figure(figsize=(7,5))
sns.scatterplot(
    data=df_focus,
    x='% Adults Reporting Currently Smoking',
    y='% Physically Inactive',
    size='Years of Potential Life Lost Rate',
    hue='Percent_Rural',
    palette='coolwarm', sizes=(50,400), alpha=0.8
)
plt.title("Behavioral Determinants & Mortality (Bubble = Years of Life Lost)", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


**Behavioral Determinants & Mortality (bubble plot)**

* As **smoking ↑**, **physical inactivity ↑** almost linearly. Bubbles (Years of Life Lost) generally get bigger in the top-right. Warmer colors (more rural) cluster higher too.

* Places where smoking is “normal” also have more inactivity—**reinforcing social norms**. In those same places, people **lose more years of life**, especially when counties are more rural (access barriers).

* **Hidden determinant angle**: Social norms (behavior clustering) + rural access constraints → less preventive care → higher mortality.

In [ ]:
# Hidden Determinant Proxies in Action

scaler = MinMaxScaler()
df_focus['Health_Literacy_Index'] = scaler.fit_transform(
    df_focus[['% Some College', '% Completed High School', 'Percent_Broadband_Access']]
).mean(axis=1)

plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df_focus,
    x='Health_Literacy_Index',
    y='Years of Potential Life Lost Rate',
    hue='Percent_Rural',
    palette='coolwarm',
    s=100, alpha=0.8
)
plt.title("Proxy for Health Literacy vs Mortality", fontsize=12)
plt.xlabel("Health Literacy / Access Index")
plt.ylabel("Years of Potential Life Lost Rate")
plt.show()


**Health Literacy / Access Index vs Mortality**

* As the **index increases** (built from education + broadband), **Years of Life Lost generally decreases. More-rural counties tend to sit left (low index) with higher mortality.

* Better education and broadband → **higher health literacy & access → fewer years of life lost**.

* **Hidden determinant angle**: “Health literacy/access” is acting like a **mediator** between structural factors (education, connectivity) and outcomes.

In [ ]:
# outlier & cluster examination
X = df_focus[['Median Household Income','% Some College','Percent_Broadband_Access','% Adults Reporting Currently Smoking','Percent_Rural']]
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(x=pca_coords[:,0], y=pca_coords[:,1], hue=df_focus['Years of Potential Life Lost Rate'], palette='coolwarm', s=80)
plt.title("County Clusters by Hidden Determinant Profiles", fontsize=12)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

**Principal Component Analysis (PCA): County clusters by hidden-determinant profiles**
* Counties separate along 2 axes built from **income**, **education**, **broadband**, **rurality**, **smoking**. Mortality shades form a gradient across the space.
  - Each dot = a county
  - X-axis (PC1) ≈ overall structural advantage (income + education + broadband).
  - Y-axis (PC2) ≈ behavioral/rural risk contrast (smoking + inactivity + rurality).
  - The color gradient (Years of Life Lost) shows how outcomes shift along those hidden dimensions.

* Counties **naturally group** into risk profiles (e.g., “connected & educated” vs “rural & behavior-risk”), and mortality tracks those profiles.

* **Hidden determinant angle**: The *combination* of structural + behavioral context is what drives the outcome—exactly the “hidden determinants” layer we’re surfacing.

##### **Phase 4.4 - Clustering: Finding County "Types"**

**Goal:**  Group West Virginia's 55 counties into profiles based on their social, economic, behavioral, and access characteristics.

Each cluster will represent a "county type." For example:
- Cluster 1: Connected & Educated
- cluster 2: Economically Strained and Behaviorally At Risk
- Cluster 3: Rural Isolation and Limited Access

In [ ]:
##### **Phase 4.4 - Clustering: Finding County "Types"**

# Choose key structural, behavioral, and access indicators
cluster_features = [
    'Median Household Income',
    '% Children in Poverty',
    '% Some College',
    'Percent_Broadband_Access',
    '% Adults Reporting Currently Smoking',
    '% Physically Inactive',
    'Percent_Rural',
    'Primary Care Physicians Rate'
]

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_focus[cluster_features])

In [ ]:
# Choose number of clusters (we’ll start with 3)
kmeans = KMeans(n_clusters=3, random_state=42)
df_focus.loc[:, 'Cluster'] = kmeans.fit_predict(X_scaled)

# View average profile per cluster
cluster_profiles = df_focus.groupby('Cluster')[cluster_features].mean().round(2)
cluster_profiles


In [ ]:
# visualize the clusters
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=df_focus,
    x='Health_Literacy_Index',
    y='Years of Potential Life Lost Rate',
    hue='Cluster',
    palette=['gold', 'lightgreen', 'firebrick'],
    s=120, alpha=0.8
)
plt.title("County Clusters by Health Literacy and Mortality", fontsize=12)
plt.show()



- **Cluster 1** - 🟩 **Green**: top-right (high literacy, low mortality)
- **Cluster 0** - 🟨 **Gold**: middle band
- **Cluster 2** - 🟥 **Red**: lower-left (low literacy, high mortality)

##### **Phase 4.5 - Mapping: Seeing Where These Types Live**

In [ ]:
df_final.columns[:10]


In [ ]:
# Add FIPS to df_focus
# make a safe copy
df_focus = df_focus.copy()

# Attach and format FIPS (5-digit strings with leading zeros)
df_focus['FIPS'] = df_final['FIPS'].astype(str).str.zfill(5)

In [ ]:
# --- Load county boundaries GeoJSON ---
url = 'https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json'
geojson_data = requests.get(url).json()


In [ ]:
# --- Create choropleth map ---
name_map = {
    0: 'Rural Behavioral Risk',
    1: 'Connected & Resilient',
    2: 'Economically Strained & Isolated'
}

df_focus['Cluster_Name'] = df_focus['Cluster'].map(name_map)

# Add 'County' column to df_focus for hover information
df_focus['County'] = df_final['County']


fig = px.choropleth(
    df_focus,
    geojson=geojson_data,
    locations='FIPS',
    color='Cluster_Name',
    color_discrete_map={
        'Rural Behavioral Risk': '#FFD700',
        'Connected & Resilient': '#2E8B57',
        'Economically Strained & Isolated': '#B22222'
    },
    scope='usa',
    title='West Virginia County Types by Health Determinant Profiles',
    hover_name='County'
)
fig.update_geos(fitbounds="locations", visible=False)
fig.show()

In [ ]:
# normalize the Health Literacy index
scaler = MinMaxScaler()
df_focus['HL_scaled'] = scaler.fit_transform(df_focus[['Health_Literacy_Index']])

In [ ]:
# build the base map (cluster layer)


# Load geojson (if not already loaded)
geojson_data = requests.get(
    'https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json'
).json()

cluster_colors = {
    'Rural Behavioral Risk': '#FFD700',
    'Connected & Resilient': '#2E8B57',
    'Economically Strained & Isolated': '#B22222'
}

fig = px.choropleth(
    df_focus,
    geojson=geojson_data,
    locations='FIPS',
    color='Cluster_Name',
    color_discrete_map=cluster_colors,
    scope='usa',
    hover_name='County',
    hover_data={
        'Health_Literacy_Index': ':.2f',
        'Years of Potential Life Lost Rate': ':.0f'
    },
    title='WV County Clusters + Health Literacy Overlay'
)


In [ ]:
# Add the Health Literacy Overlay
# Base cluster layer
cluster_layer = go.Choroplethmapbox(
    geojson=geojson_data,
    locations=df_focus['FIPS'],
    z=df_focus['Cluster'].astype(int),
    colorscale=[[0, '#FFD700'], [0.5, '#2E8B57'], [1, '#B22222']],
    showscale=False,
    marker_opacity=0.9,
    marker_line_width=0.5
)

# Overlay literacy gradient layer
literacy_layer = go.Choroplethmapbox(
    geojson=geojson_data,
    locations=df_focus['FIPS'],
    z=df_focus['HL_scaled'],
    colorscale='Blues',
    zmin=0, zmax=1,
    showscale=True,
    marker_opacity=0.45,   # semi-transparent overlay
    marker_line_width=0
)

fig = go.Figure(cluster_layer)
fig.add_trace(literacy_layer)

fig.update_layout(
    mapbox_style='carto-positron',
    mapbox_zoom=6,
    mapbox_center={"lat": 38.6, "lon": -80.6},
    margin={"r":0,"t":40,"l":0,"b":0},
    title_text="WV County Clusters with Health Literacy Overlay (Blue = Higher Literacy)"
)

fig.show()


**Quick Read of the Visualization**

| Layer           | Meaning                                           | What to Look For                                                                                                                 |
| --------------- | ------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| 🎨 Base Colors  | County cluster type (social/economic environment) | The *type* of structural determinant shaping the area — rural behavioral risk, economically strained, or connected/resilient.    |
| 💙 Blue Overlay | Health Literacy Index (higher = darker)           | The *within-cluster gradient* — counties of the same type that differ in digital access, literacy, and preventive care behavior. |


> **Interpretation Cues**
>
> - **Bright yellow + no blue** → Rural, high-risk, low literacy. These are prime targets for digital-literacy and preventive-care campaigns.
>
> - **Red + deep blue** → Economically strained yet high-literacy. These places might respond well to telehealth outreach.
>
> - **Green + blue** → Connected & Resilient strongholds — benchmark regions for best-practice modeling.

In [ ]:
# Enhance the Cluster + Literacy Map

# Ensure required columns exist
# df_focus must include: FIPS (str,zfill(5)), County, Cluster (0/1/2), Cluster_Name, Health_Literacy_Index

# A. Scale literacy to [0,1] for overlay intensity
scaler = MinMaxScaler()
df_focus['HL_scaled'] = scaler.fit_transform(df_focus[['Health_Literacy_Index']])

# B. Stable colors + names for clusters
name_map = {0:'Rural Behavioral Risk', 1:'Connected & Resilient', 2:'Economically Strained & Isolated'}
df_focus['Cluster_Name'] = df_focus['Cluster'].map(name_map)

cluster_colors = {
    'Rural Behavioral Risk': '#FFD700',        # yellow
    'Connected & Resilient': '#2E8B57',        # green
    'Economically Strained & Isolated': '#B22222'  # red
}

# For mapbox layer, build an ordered list matching numeric z-values 0,1,2
cluster_order = ['Rural Behavioral Risk','Connected & Resilient','Economically Strained & Isolated']
cluster_color_list = [cluster_colors[k] for k in cluster_order]

# C. Load county GeoJSON
geojson_data = requests.get(
    'https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json'
).json()


**Build the two layers**
- **Layer 1**: clusters (categorical base)
- **Layer 2**: literacy gradient (semi-transparent blue overlay)

In [ ]:
# build two layers
# Build a mapping from cluster int -> index in cluster_order
int_to_idx = {i: cluster_order.index(name_map[i]) for i in [0,1,2]}
z_cluster = df_focus['Cluster'].map(int_to_idx).astype(float)

# A) Base cluster layer (categorical via a segmented colorscale)
cluster_colorscale = [
    [0/3, cluster_color_list[0]], [1/3 - 1e-6, cluster_color_list[0]],
    [1/3, cluster_color_list[1]], [2/3 - 1e-6, cluster_color_list[1]],
    [2/3, cluster_color_list[2]], [1.0,        cluster_color_list[2]],
]

cluster_layer = go.Choroplethmapbox(
    geojson=geojson_data,
    locations=df_focus['FIPS'],
    z=z_cluster,                    # 0,1,2 via int_to_idx
    colorscale=cluster_colorscale,
    showscale=False,
    marker_opacity=0.95,
    marker_line_width=0.5,
    featureidkey="id",
    hoverinfo='skip'                # hover handled by overlay for richer text
)

# B) Literacy overlay (continuous + transparent)
# customdata for rich hover: County, Cluster_Name, HL, YPLL
customdata = df_focus[['County','Cluster_Name','Health_Literacy_Index',
                       'Years of Potential Life Lost Rate']].values

literacy_layer = go.Choroplethmapbox(
    geojson=geojson_data,
    locations=df_focus['FIPS'],
    z=df_focus['HL_scaled'],
    colorscale='Blues',
    zmin=0, zmax=1,
    showscale=True,
    colorbar=dict(title='Health Literacy / Access (0–1)'),
    marker_opacity=0.45,
    marker_line_width=0,
    featureidkey="id",
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"                      # County
        "Type: %{customdata[1]}<br>"                       # Cluster_Name
        "Health Literacy: %{customdata[2]:.2f}<br>"        # HL index original scale
        "Years of Life Lost: %{customdata[3]:.0f}<extra></extra>"
    ),
    customdata=customdata
)

fig = go.Figure(cluster_layer)
fig.add_trace(literacy_layer)

fig.update_layout(
    mapbox_style='carto-positron',
    mapbox_zoom=6,
    mapbox_center={"lat": 38.6, "lon": -80.6},
    margin={"r":0,"t":50,"l":0,"b":0},
    title_text="WV County Clusters with Health Literacy Overlay (Blue = Higher Literacy)"
)


In [ ]:
# Add a clean cluster legend (annotations)
# Legend swatches (bottom-right-ish). Tweak x,y to move it.
#
# Define legend items (order and colors)
def add_cluster_legend(fig, x0=0.74, y0=0.04, row_h=0.065):
    """
    Add a tidy legend box in paper coords.
    x0,y0 = bottom-left corner. row_h controls spacing.
    """
    items = [
        ("Connected & Resilient", "#2E8B57"),
        ("Rural Behavioral Risk", "#FFD700"),
        ("Economically Strained & Isolated", "#B22222"),
    ]
    # panel (width ~ 0.24, height ~ rows + title)
    width, height = 0.24, (len(items)+1)*row_h + 0.02
    fig.add_shape(
        type="rect", xref="paper", yref="paper",
        x0=x0, y0=y0, x1=x0+width, y1=y0+height,
        fillcolor="rgba(255,255,255,0.92)", line=dict(color="rgba(0,0,0,0.25)", width=1)
    )
    # title
    fig.add_annotation(
        xref="paper", yref="paper",
        x=x0+0.01, y=y0+height-0.02,
        text="<b>County Cluster Type</b>", showarrow=False,
        font=dict(size=12, color="#222"), align="left"
    )
    # rows
    y = y0+height-0.02-row_h
    for label, color in items:
        # swatch
        fig.add_shape(
            type="rect", xref="paper", yref="paper",
            x0=x0+0.01, x1=x0+0.05, y0=y-0.012, y1=y+0.012,
            fillcolor=color, line=dict(color="rgba(0,0,0,0.4)", width=0.5)
        )
        # text
        fig.add_annotation(
            xref="paper", yref="paper",
            x=x0+0.06, y=y, text=label, showarrow=False,
            font=dict(size=12, color="#111"), align="left"
        )
        y -= row_h

# call it after you’ve created `fig`
add_cluster_legend(fig, x0=0.73, y0=0.03)  # tweak x0/y0 if you want it moved


fig.show()

##### **4.8 - Quantitative Comparison of County Cluster Profiles**

In [ ]:
# Step 4.8.1 — Compute average values per cluster
profile_features = [
    'Median Household Income', '% Children in Poverty', '% Some College',
    '% Adults Reporting Currently Smoking', '% Physically Inactive',
    'Percent_Broadband_Access', 'Primary Care Physicians Rate',
    'Health_Literacy_Index', 'Years of Potential Life Lost Rate'
]

# Group by cluster label
cluster_summary = (
    df_focus.groupby('Cluster_Name')[profile_features]
    .mean()
    .round(2)
    .sort_index()
)

# Display table
display(cluster_summary)


In [ ]:
# --- 1. Copy summary ---
scaled_summary = cluster_summary.copy()

# --- 2. Normalize each column (Min–Max scaling 0–1) ---
scaler = MinMaxScaler()
scaled_summary[scaled_summary.columns] = scaler.fit_transform(scaled_summary)

# --- 3. Plot scaled bar chart ---
scaled_summary.T.plot(kind='bar', figsize=(12,6))
plt.title('Normalized Comparison of Determinants by County Cluster')
plt.ylabel('Scaled Value (0 = lowest, 1 = highest)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='County Cluster Type')
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()


**Analysis** - *Normalized Bar Chart*

This plot compares each cluster’s scaled performance (0–1) on each determinant.

| Determinant                               | What the chart shows                                                           | Interpretation                                                    |
| ----------------------------------------- | ------------------------------------------------------------------------------ | ----------------------------------------------------------------- |
| **Median Household Income**               | Connected & Resilient counties (blue) top the scale.                           | Economically strong, higher literacy and access.                  |
| **% Children in Poverty**                 | Strained counties (orange) peak at 1.0 (worst).                                | Poverty dominates outcomes in those regions.                      |
| **% Some College**                        | Blue highest again → better education access.                                  | Education ties closely to literacy.                               |
| **% Smoking / % Inactive**                | Orange leads → behavioral risk concentrated in economically strained counties. | Health habits amplify structural disadvantages.                   |
| **Broadband / PCP Rate / Literacy Index** | Blue dominates → robust access + digital literacy.                             | Infrastructure drives information access and prevention capacity. |
| **Years of Potential Life Lost**          | Orange high → worse mortality outcomes.                                        | Confirms socioeconomic & behavioral vulnerability link.           |

- 🟦 Blue (**Connected & Resilient**): high access, literacy, infrastructure
- 🟩 Green (**Rural Behavioral Risk**): moderate across the board
- 🟧 Orange (**Economically Strained & Isolated**): lowest socioeconomic + health literacy, highest mortality

In [ ]:
# --- 4. Radar Chart ---
radar = px.line_polar(
    scaled_summary.reset_index().melt(id_vars='Cluster_Name'),
    r='value', theta='variable', color='Cluster_Name',
    line_close=True, template='plotly_white',
    title="Scaled Cluster Profiles Across Determinants"
)
radar.update_traces(fill='toself', opacity=0.6)
radar.show()


**Analysis** - *Radar Chart — “Hidden Determinants Shape Profile*

This chart beautifully captures the shape of each county type:

| Cluster                                 | Shape & Meaning                                                                                       |
| --------------------------------------- | ----------------------------------------------------------------------------------------------------- |
| 🟦 **Connected & Resilient**            | Wide, balanced polygon → Strong across most determinants. “Healthy networked systems.”                |
| 🟧 **Economically Strained & Isolated** | Sharp inward spikes → multiple weaknesses (poverty, literacy, healthcare). “Structural deprivation.”  |
| 🟩 **Rural Behavioral Risk**            | Uneven shape → decent access but behavioral health risk remains. “Norms gap, not infrastructure gap.” |

The *“hidden determinants”* (like health literacy and social behavior) now visibly mediate between structure (income, access) and outcomes (mortality).

In [ ]:
# Save the DataFrames to CSVs first
df_final.to_csv('df_final_cleaned.csv', index=False)
df_focus.to_csv('df_focus.csv', index=False)
cluster_summary.to_csv('cluster_summary.csv', index=False)

# Now download them
from google.colab import files
files.download('df_final_cleaned.csv')
files.download('df_focus.csv')
files.download('cluster_summary.csv')


## **Phase 5 Overview: Predictive Analytics Model Development**
- Bridge your exploratory insights to a model that forecasts care-seeking disparities based on social determinants of health (SDOH).

In [ ]:
df = pd.read_csv('wv_trubridge_cleaned_phase3.csv')
df.shape, df.columns[:10]

In [ ]:
df.isna().sum().sum()



In [ ]:
df.describe()